# 4 — Cell-level benchmark: Scanpy PCA versus frozen scGPT

## Question

On exactly the same cells, which representation better preserves known broad
cell identities and fine CD4/CD8 states?

Evidence includes UMAPs, label silhouette, kNN label accuracy, Leiden ARI/NMI,
and platform silhouette. UMAP appearance alone is not treated as proof.

In [ ]:
from pathlib import Path
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures/demo"
TABLES = RESULTS / "tables"
EMBEDDINGS = RESULTS / "embeddings"
for directory in (FIGURES, TABLES, EMBEDDINGS):
    directory.mkdir(parents=True, exist_ok=True)
RAW_PATH = DATA / "processed/GSE205335_phase1_raw_counts.h5ad"
RANDOM_STATE = 0

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.neighbors import KNeighborsClassifier

PCA_PATH = EMBEDDINGS / "GSE205335_10sample_scanpy_pca_v1.h5ad"
SCGPT_PATH = EMBEDDINGS / "GSE205335_scgpt_frozen_v1.h5ad"
assert PCA_PATH.exists() and SCGPT_PATH.exists(), "Run Notebooks 2 and 3 first"
pca = sc.read_h5ad(PCA_PATH)
scgpt = sc.read_h5ad(SCGPT_PATH)
assert pca.obs_names.equals(scgpt.obs_names)
label_columns = ["lineage.total", "lineage.sub", "celltype"]
scgpt.obs[label_columns] = pca.obs.loc[scgpt.obs_names, label_columns].copy()
print("Matched cells:", pca.n_obs)

## 1. Build comparable graphs and UMAPs

In [ ]:
views = {"Scanpy_PCA": pca.copy(), "frozen_scGPT": scgpt.copy()}
for name, obj in views.items():
    sc.pp.neighbors(obj, use_rep="X", n_neighbors=15, random_state=RANDOM_STATE)
    sc.tl.umap(obj, random_state=RANDOM_STATE)
    sc.tl.leiden(obj, resolution=0.6, key_added="cluster", random_state=RANDOM_STATE,
                 flavor="igraph", n_iterations=2, directed=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (name, obj) in zip(axes, views.items()):
    sc.pl.umap(obj, color="lineage.total", title=name, ax=ax, show=False)
fig.tight_layout()
fig.savefig(FIGURES / "cell_level_broad_pca_vs_scgpt.png", dpi=220, bbox_inches="tight")
plt.show()

## 2. Quantitative broad-cell benchmark

Silhouette uses a fixed 5,000-cell subsample to avoid constructing a full
29,614 × 29,614 distance matrix. kNN is evaluated by sample-held-out folds, so
cells from the test sample never appear in training.

In [ ]:
def held_out_sample_knn(matrix, labels, samples, k=15):
    predictions = np.empty(len(labels), dtype=object)
    for sample in pd.unique(samples):
        test = samples == sample
        model = KNeighborsClassifier(n_neighbors=k, weights="distance", metric="cosine", n_jobs=-1)
        model.fit(matrix[~test], labels[~test])
        predictions[test] = model.predict(matrix[test])
    return float(np.mean(predictions == labels))

def benchmark(name, obj, labels, scope):
    matrix = np.asarray(obj.X, dtype=np.float32)
    labels = np.asarray(labels).astype(str)
    samples = obj.obs["Sample"].astype(str).to_numpy()
    platform = obj.obs["Platform"].astype(str).to_numpy()
    return {
        "scope": scope, "representation": name, "n_cells": len(labels),
        "label_silhouette": silhouette_score(matrix, labels, metric="cosine", sample_size=min(5000, len(labels)), random_state=0),
        "sample_held_out_knn_accuracy": held_out_sample_knn(matrix, labels, samples),
        "leiden_ARI": adjusted_rand_score(labels, obj.obs["cluster"].astype(str)),
        "leiden_NMI": normalized_mutual_info_score(labels, obj.obs["cluster"].astype(str)),
        "platform_silhouette": silhouette_score(matrix, platform, metric="cosine", sample_size=min(5000, len(labels)), random_state=0),
    }

rows = [benchmark(name, obj, obj.obs["lineage.total"], "broad_all_cells") for name, obj in views.items()]

## 3. Fine-state CD4 and CD8 benchmark

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for row_index, lineage in enumerate(["CD4+ T cells", "CD8+ T cells"]):
    for col_index, (name, obj) in enumerate(views.items()):
        mask = obj.obs["lineage.sub"].astype(str).eq(lineage)
        subset = obj[mask].copy()
        sc.pp.neighbors(subset, use_rep="X", n_neighbors=15, random_state=RANDOM_STATE)
        sc.tl.umap(subset, random_state=RANDOM_STATE)
        sc.tl.leiden(subset, resolution=0.8, key_added="cluster", random_state=RANDOM_STATE,
                     flavor="igraph", n_iterations=2, directed=False)
        sc.pl.umap(subset, color="celltype", title=f"{lineage}: {name}", ax=axes[row_index, col_index], show=False)
        rows.append(benchmark(name, subset, subset.obs["celltype"], lineage))
fig.tight_layout()
fig.savefig(FIGURES / "cell_level_CD4_CD8_pca_vs_scgpt.png", dpi=220, bbox_inches="tight")
plt.show()

metrics = pd.DataFrame(rows)
display(metrics)
metrics.to_csv(TABLES / "cell_level_pca_vs_scgpt_metrics.csv", index=False)

## Reading the evidence

- Higher label silhouette, held-out kNN accuracy, ARI and NMI are better.
- Platform silhouette should be near zero; a large positive value suggests a
  technical platform effect.
- A credible conclusion reports where scGPT wins, ties, or loses rather than
  assuming the foundation model must be superior.